# 🔑 Notebook 2: Idempotency Keys

The client generates a unique key (a UUID) for each *logical* request. The server stores `key -> result` the first time. Subsequent retries with the same key get the original result back — no side effect.

This is how Stripe, AWS, GitHub, and most modern APIs handle retries safely.


## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 In-memory implementation

In [ ]:
import uuid

class PaymentService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}  # key -> stored result

    def charge(self, key, account, amount):
        if key in self.idem:
            print(f'  ↩ replay for {key[:8]}…')
            return self.idem[key]
        self.balances[account] -= amount
        result = {'ok': True, 'balance': self.balances[account], 'charged': amount}
        self.idem[key] = result
        return result

svc = PaymentService()
k = str(uuid.uuid4())
print('1st:', svc.charge(k, 'alice', 10))
print('2nd:', svc.charge(k, 'alice', 10))
print('3rd:', svc.charge(k, 'alice', 10))
print('Alice ends with', svc.balances['alice'], '— charged exactly once ✅')


Now the same key replays the *cached* response. A different key would charge again — that's correct, because it's a *different* logical request.